# Bab 9 — Teknik balancing pada data latih

Memerlukan folder output/ dari notebook 01.

## Kode 9.1

In [ ]:
import json, time
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, imblearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

# Muat hasil Bab 8. Di bab ini HANYA data latih yang dipakai.
X_train = pd.read_csv("output/X_train.csv")
y_train = pd.read_csv("output/y_train.csv")["y"].to_numpy()
nama_kelas = pd.read_csv("output/pemetaan_label.csv")["label"].tolist()

# Sampler berbasis jarak memerlukan fitur numerik berskala (keputusan bab ini; lihat teks)
kolom_num = X_train.select_dtypes(include="number").columns.tolist()
praproses = Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())])
Z_train = praproses.fit_transform(X_train[kolom_num])    # dipasang pada data latih saja
print("Fitur numerik:", len(kolom_num), "| bentuk Z_train:", Z_train.shape)

def distribusi(y):
    c = Counter(y)
    return {nama_kelas[k]: c[k] for k in sorted(c)}

hasil, waktu = {"Asli": distribusi(y_train)}, {}

def jalankan(nama, sampler):
    """Jalankan sampler HANYA pada data latih; catat distribusi dan waktu."""
    t0 = time.perf_counter()
    Zr, yr = sampler.fit_resample(Z_train, y_train)
    waktu[nama] = time.perf_counter() - t0
    hasil[nama] = distribusi(yr)
    print(f"{nama}: total {len(yr)} sampel (asli {len(y_train)}), {waktu[nama]:.2f} dtk")
    return Zr, yr

def gambar(nama):
    d = pd.DataFrame({"Asli": hasil["Asli"], nama: hasil[nama]})
    ax = d.plot.barh(figsize=(9, 5), logx=True, color=["#9CA3AF", "#2E9C93"])
    ax.invert_yaxis()
    ax.set_xlabel("Jumlah sampel (skala logaritmik)")
    plt.tight_layout()
    plt.savefig(f"output/g_{nama.replace(' ', '_')}.png", dpi=200)
    plt.show()

jumlah = Counter(y_train)
print("Distribusi latih asli:", hasil["Asli"])

## Kode 9.2

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# a) Pengaturan bawaan: semua kelas selain yang terkecil diturunkan ke jumlah kelas terkecil
_ = jalankan("RUS bawaan", RandomUnderSampler(random_state=RANDOM_STATE))

# b) Target parsial: tiap kelas dibatasi paling banyak BATAS sampel
BATAS = 2000                                   # CEK: sesuaikan dengan distribusi Anda
target_rus = {k: min(v, BATAS) for k, v in jumlah.items()}
_ = jalankan("RUS parsial", RandomUnderSampler(sampling_strategy=target_rus,
                                               random_state=RANDOM_STATE))
print(hasil["RUS bawaan"])
print(hasil["RUS parsial"])
gambar("RUS parsial")

## Kode 9.3

In [ ]:
from imblearn.over_sampling import RandomOverSampler

# a) Bawaan: semua kelas selain yang terbesar dinaikkan ke jumlah kelas terbesar
_ = jalankan("ROS bawaan", RandomOverSampler(random_state=RANDOM_STATE))

# b) Target parsial: tiap kelas dinaikkan paling tidak sampai TARGET sampel
TARGET = 2000                                  # CEK
target_ros = {k: max(v, TARGET) for k, v in jumlah.items()}
Zr, yr = jalankan("ROS parsial", RandomOverSampler(sampling_strategy=target_ros,
                                                   random_state=RANDOM_STATE))
# ROS menggandakan baris asli: jumlah baris unik tidak bertambah
print("Baris unik setelah ROS parsial:", len(np.unique(Zr, axis=0)),
      "| baris unik data latih asli:", len(np.unique(Z_train, axis=0)))
gambar("ROS parsial")

## Kode 9.4

In [ ]:
from imblearn.over_sampling import SMOTE

# k_neighbors harus lebih kecil dari jumlah sampel kelas terkecil (subbab 7.7)
jumlah_min = min(jumlah.values())
K = max(1, min(5, jumlah_min - 1))
print("Sampel kelas terkecil:", jumlah_min, "| k_neighbors dipakai:", K)

# a) Bawaan: semua kelas selain yang terbesar dinaikkan ke jumlah kelas terbesar
_ = jalankan("SMOTE bawaan", SMOTE(k_neighbors=K, random_state=RANDOM_STATE))

# b) Target parsial (TARGET dan target_ros sudah didefinisikan pada Kode 9.3)
_ = jalankan("SMOTE parsial", SMOTE(sampling_strategy=target_ros, k_neighbors=K,
                                    random_state=RANDOM_STATE))
gambar("SMOTE parsial")

## Kode 9.5

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE, ADASYN

# Borderline-SMOTE (varian 1): sampel sintetis hanya dari sampel "bahaya"
bl = BorderlineSMOTE(kind="borderline-1", k_neighbors=K, m_neighbors=10,
                     sampling_strategy=target_ros, random_state=RANDOM_STATE)
_ = jalankan("Borderline-SMOTE", bl)
bahaya = {nama_kelas[k]: len(v) for k, v in bl.in_danger_indices.items()}
print("Sampel 'bahaya' per kelas yang dinaikkan:", bahaya)

# ADASYN: jumlah akhir tidak selalu tepat sama dengan target; dapat gagal
try:
    _ = jalankan("ADASYN", ADASYN(sampling_strategy=target_ros, n_neighbors=K,
                                  random_state=RANDOM_STATE))
except (RuntimeError, ValueError) as e:
    print("ADASYN gagal:", e)

for nama in ("SMOTE parsial", "Borderline-SMOTE", "ADASYN"):
    if nama in hasil:
        selisih = {k: hasil[nama][k] - target_ros[i]
                   for i, k in enumerate(nama_kelas) if i in target_ros}
        print(nama, "| selisih jumlah akhir terhadap target per kelas:", selisih)
gambar("Borderline-SMOTE")

## Kode 9.6

In [ ]:
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

# SMOTE dahulu, lalu Tomek Links pada SEMUA kelas (bawaan SMOTETomek)
st = SMOTETomek(smote=SMOTE(sampling_strategy=target_ros, k_neighbors=K,
                            random_state=RANDOM_STATE),
                random_state=RANDOM_STATE)
_ = jalankan("SMOTE-Tomek", st)

# Sampel yang dibuang tahap Tomek = selisih terhadap SMOTE parsial (SMOTE-nya identik)
dibuang = {k: hasil["SMOTE parsial"][k] - hasil["SMOTE-Tomek"][k] for k in nama_kelas}
print("Sampel dibuang tahap Tomek per kelas:", dibuang)
print("Total dibuang:", sum(dibuang.values()))
gambar("SMOTE-Tomek")

## Kode 9.7

In [ ]:
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import EditedNearestNeighbours

def smote_parsial():
    return SMOTE(sampling_strategy=target_ros, k_neighbors=K, random_state=RANDOM_STATE)

# a) ENN bawaan: kind_sel="all" pada semua kelas
_ = jalankan("SMOTE-ENN", SMOTEENN(smote=smote_parsial(), random_state=RANDOM_STATE))

# b) ENN lebih ringan: kind_sel="mode"
enn_mode = EditedNearestNeighbours(sampling_strategy="all", kind_sel="mode")
_ = jalankan("SMOTE-ENN mode", SMOTEENN(smote=smote_parsial(), enn=enn_mode,
                                        random_state=RANDOM_STATE))

for nama in ("SMOTE-ENN", "SMOTE-ENN mode"):
    dibuang = {k: hasil["SMOTE parsial"][k] - hasil[nama][k] for k in nama_kelas}
    print(nama, "| total dibuang:", sum(dibuang.values()), "|", dibuang)
gambar("SMOTE-ENN")

## Kode 9.8

In [ ]:
# Tabel distribusi semua metode (baris = kelas, kolom = metode)
tabel = pd.DataFrame(hasil).fillna(0).astype(int)
print(tabel)
tabel.to_csv("output/dist_sebelum_sesudah.csv")

# Ringkasan per metode: total, kelas terbesar dan terkecil, IR, waktu
ringkas = pd.DataFrame({
    "total": tabel.sum(),
    "terbesar": tabel.max(),
    "terkecil": tabel.min(),
})
ringkas["IR"] = (ringkas["terbesar"] / ringkas["terkecil"]).round(2)
ringkas["faktor_ukuran"] = (ringkas["total"] / ringkas.loc["Asli", "total"]).round(2)
ringkas["waktu_dtk"] = pd.Series(waktu).round(3)
print(ringkas)
ringkas.to_csv("output/ringkasan_metode.csv")

# Grafik gabungan: peta panas jumlah per kelas (baris) untuk tiap metode (kolom).
# Warna memakai log10 agar kelas besar dan kecil terbaca bersama; angka asli ditulis di sel.
nilai = tabel.to_numpy()
fig, ax = plt.subplots(figsize=(8, 5.2))
im = ax.imshow(np.log10(nilai + 1), cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(tabel.shape[1]))
ax.set_xticklabels(tabel.columns, rotation=40, ha="right", fontsize=7)
ax.set_yticks(range(tabel.shape[0]))
ax.set_yticklabels(tabel.index, fontsize=7)
for i in range(nilai.shape[0]):
    for j in range(nilai.shape[1]):
        warna = "white" if np.log10(nilai[i, j] + 1) > 3.3 else "black"
        ax.text(j, i, f"{nilai[i, j]:,}".replace(",", "."), ha="center", va="center",
                fontsize=5.5, color=warna)
fig.colorbar(im, ax=ax, label="log10 (jumlah sampel + 1)", fraction=0.04, pad=0.02)
plt.tight_layout()
plt.savefig("output/g_perbandingan_metode.png", dpi=200)
plt.show()

## Kode 9.9

In [ ]:
# Eksperimen target jumlah sampel: 25%, 50%, 100% dari kelas mayoritas
maj = max(jumlah.values())
BATAS_UKURAN = 1_000_000        # CEK: batas jumlah baris yang masih muat di memori dan waktu Anda
baris = []
for frac in (0.25, 0.50, 1.00):
    target = {k: max(v, int(frac * maj)) for k, v in jumlah.items()}
    ukuran = sum(target.values())
    if ukuran > BATAS_UKURAN:
        print(f"Target {frac:.0%} dilewati: {ukuran} baris melebihi BATAS_UKURAN")
        baris.append({"target": f"{frac:.0%} mayoritas", "total": ukuran, "dijalankan": False})
        continue
    t0 = time.perf_counter()
    Zr, yr = SMOTE(sampling_strategy=target, k_neighbors=K,
                   random_state=RANDOM_STATE).fit_resample(Z_train, y_train)
    baris.append({"target": f"{frac:.0%} mayoritas", "total": len(yr),
                  "faktor_ukuran": round(len(yr) / len(y_train), 2),
                  "terkecil": min(Counter(yr).values()),
                  "waktu_dtk": round(time.perf_counter() - t0, 2), "dijalankan": True})
tabel_target = pd.DataFrame(baris)
print(tabel_target)
tabel_target.to_csv("output/dampak_target.csv", index=False)

ax = tabel_target.plot.bar(x="target", y="total", legend=False, color="#E07B39", figsize=(6, 4))
ax.set_ylabel("Jumlah baris data latih setelah balancing")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("output/g_dampak_target.png", dpi=200)
plt.show()

## Kode 9.10

In [ ]:
import os
os.makedirs("output/hasil_balancing", exist_ok=True)

# Simpan satu hasil sebagai array terkompresi. Konvensi nama: metode_target_seed.npz
Zr, yr = SMOTE(sampling_strategy=target_ros, k_neighbors=K,
               random_state=RANDOM_STATE).fit_resample(Z_train, y_train)
nama_file = f"output/hasil_balancing/smote_target{TARGET}_seed{RANDOM_STATE}.npz"
np.savez_compressed(nama_file, Z=Zr, y=yr, kolom=np.array(kolom_num))
print("Tersimpan:", nama_file, "| ukuran:", round(os.path.getsize(nama_file) / 1e6, 2), "MB")

# Cek bahwa berkas dapat dibaca kembali
muat = np.load(nama_file)
print("Dibaca ulang:", muat["Z"].shape, muat["y"].shape)

## Kode 9.11

In [ ]:
# Dokumentasi: konfigurasi, versi, seed, dan waktu eksekusi disimpan sebagai JSON dan CSV
konfigurasi = {
    "versi": {lib.__name__: lib.__version__ for lib in (np, pd, sklearn, imblearn)},
    "random_state": RANDOM_STATE,
    "fitur": "numerik berskala (imputer median dan StandardScaler dipasang pada data latih)",
    "k_neighbors": int(K),
    "BATAS_rus": BATAS,
    "TARGET_oversampling": TARGET,
    "waktu_detik": {k: round(v, 3) for k, v in waktu.items()},
    "distribusi": hasil,
}
with open("output/konfigurasi_bab9.json", "w") as f:
    json.dump(konfigurasi, f, indent=2, ensure_ascii=False, default=int)

log = pd.DataFrame({"waktu_dtk": pd.Series(waktu).round(3),
                    "total_sampel": pd.DataFrame(hasil).drop(columns="Asli").sum().astype(int)})
log["random_state"] = RANDOM_STATE
log["k_neighbors"] = K
print(log)
log.to_csv("output/log_eksperimen_bab9.csv")